# TraceWinEnv one-step debug

This notebook runs the smallest useful TraceWinEnv check:

1. verify that SSH to `comunian@localhost` is reachable;
2. instantiate `TraceWinEnv`;
3. fix TraceWin's own Monte Carlo seed so the run uses common random numbers (CRN);
4. call `reset()`;
5. call one `step()` with zero action;
6. print the simulator result, errors, score, and files written in `calc_dir`;
7. optionally render the episode and final phase-space distribution.

It is meant for debugging TraceWin itself, not for training or policy evaluation.

**Determinism note:** `env.reset(seed=SEED)` alone only fixes the RL-level
sampling of the reset parameter perturbation (numpy RNG); it does **not**
control TraceWin's own internal Monte Carlo noise. `env.simulator.tracewin_params["random_seed"]`
is set explicitly below — the same common-random-numbers (CRN) pattern used
in `beam_optimization/config/utility/sensitivity.py` — to cancel most of that
noise. `env.simulator.num_threads` is left at its default (`None` = all
CPUs) for speed; per `sensitivity.py`, multi-threaded TraceWin runs are not
guaranteed bit-reproducible even at a fixed seed, so if you see the score
drift between identical runs, set `NUM_THREADS = 1` below to force
bit-reproducible (but slower) execution.


In [ ]:
from __future__ import annotations

import os
import subprocess
import sys
import traceback
import time
from pathlib import Path

import numpy as np


def find_repo_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "beam_optimization").is_dir():
            return candidate
    raise RuntimeError(f"Could not find repository root from {start}")


REPO_ROOT = find_repo_root(Path.cwd().resolve())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

os.environ.setdefault("MPLCONFIGDIR", "/tmp/matplotlib")

from beam_optimization.config.adige import default_params
from beam_optimization.config.paths import DEFAULT_TRACEWIN_INI
from beam_optimization.env.tracewin_env import TraceWinEnv

SEED = 123
TIMEOUT_SECONDS = 25.0
RETRIES = 0

# Determinism: TraceWin's own Monte Carlo noise is controlled by its
# random_seed CLI option (fixed on the simulator below, same CRN pattern as
# config/utility/sensitivity.py). num_threads=None runs on all CPUs for
# speed; set to 1 if runs turn out not to be reproducible at a fixed seed
# (per sensitivity.py, multi-threaded TraceWin isn't guaranteed bit-reproducible).
NUM_THREADS = None

project_file = Path(DEFAULT_TRACEWIN_INI)
calc_dir = project_file.parent / "calc_debug_env"

print(f"repo_root    = {REPO_ROOT}")
print(f"project_file = {project_file}")
print(f"calc_dir     = {calc_dir}")


In [ ]:
def list_calc_files(path: Path) -> None:
    if not path.exists():
        print(f"calc_dir does not exist: {path}")
        return
    files = sorted(p for p in path.iterdir() if p.is_file())
    if not files:
        print("calc_dir is empty")
        return
    for p in files:
        print(f"{p.name:<28} {p.stat().st_size:>12,} bytes")


def print_sim_result(label: str, info: dict) -> None:
    result = info.get("sim_result")
    print()
    print(f"[{label}]")
    print(f"score = {info.get('score')}")
    if result is None:
        print("sim_result missing")
        return
    print(f"success     = {result.success}")
    print(f"source      = {result.source}")
    print(f"score_val   = {result.score_val}")
    print(f"error       = {result.error}")
    print(f"beam_states = {None if result.beam_states is None else result.beam_states.shape}")
    print(f"metadata    = {result.metadata}")


In [ ]:
# TraceWin is launched through SSH as comunian@localhost. Check that first.
ssh_cmd = [
    "ssh", "-F", "/dev/null",
    "-o", "BatchMode=yes",
    "-o", "ConnectTimeout=5",
    "comunian@localhost",
    "echo tracewin-ssh-ok",
]

try:
    ssh_check = subprocess.run(ssh_cmd, timeout=10, capture_output=True, text=True)
    print("returncode:", ssh_check.returncode)
    print("stdout:", ssh_check.stdout.strip())
    print("stderr:", ssh_check.stderr.strip())
except Exception:
    traceback.print_exc()


In [ ]:
# Instantiate TraceWinEnv. timeout and retries are intentionally small for this debug notebook.
env = TraceWinEnv(
    project_file=str(project_file),
    calc_dir=str(calc_dir),
    max_steps=1,
    timeout=TIMEOUT_SECONDS,
    retries=RETRIES,
)

# Force determinism. TraceWinEnv does not expose random_seed/num_threads in
# its constructor, so set them directly on the simulator it built (same
# pattern as _set_seed() in config/utility/sensitivity.py):
# - tracewin_params["random_seed"] fixes TraceWin's internal Monte Carlo seed.
# - num_threads follows NUM_THREADS above (None = all CPUs, fast; 1 = bit-reproducible).
env.simulator.tracewin_params["random_seed"] = SEED
env.simulator.num_threads = NUM_THREADS

print("env created")
print("observation_space:", env.observation_space)
print("action_space:", env.action_space)
print("simulator:", type(env.simulator).__name__)
print("simulator.calc_dir:", env.simulator.calc_dir)
print("simulator.tracewin_params:", env.simulator.tracewin_params)
print("simulator.num_threads:", env.simulator.num_threads)


In [ ]:
# Run reset(): this performs the first TraceWin simulation.
try:
    t0 = time.perf_counter()
    obs, info = env.reset(seed=SEED, options={"randomize_params": False})
    elapsed = time.perf_counter() - t0
    print(f"reset elapsed: {elapsed:.2f} s")
    print("obs.shape:", obs.shape)
    print("reset_randomized:", info.get("reset_randomized"))
    print("current_params == default_params():", env._current_params == default_params())
    print("current_params:", env._current_params)
    print_sim_result("reset", info)
except Exception:
    traceback.print_exc()

print()
print("calc_dir after reset:")
list_calc_files(calc_dir)


In [ ]:
# Run exactly one step with zero action.
try:
    action = np.zeros(env.action_space.shape, dtype=np.float32)
    t0 = time.perf_counter()
    obs, reward, terminated, truncated, info = env.step(action)
    elapsed = time.perf_counter() - t0
    print(f"step elapsed: {elapsed:.2f} s")
    print("obs.shape:", obs.shape)
    print("reward:", reward)
    print("terminated:", terminated)
    print("truncated:", truncated)
    print_sim_result("step zero action", info)
except Exception:
    traceback.print_exc()

print()
print("calc_dir after step:")
list_calc_files(calc_dir)


In [ ]:
# Optional: render the episode trends and, if a .dst exists, the final beam distribution.
# The figures are saved as PNGs instead of printing the full Figure objects.
try:
    import matplotlib.pyplot as plt

    render_result = env.render(render_beam_distribution=True, max_particles=40000, bins=150)
    render_paths = {}

    for name, fig in render_result.items():
        if fig is None or not hasattr(fig, "savefig"):
            continue
        output_path = calc_dir / f"debug_render_{name}.png"
        fig.savefig(output_path, dpi=130)
        plt.close(fig)
        render_paths[name] = output_path

    if render_paths:
        print("saved render files:")
        for name, output_path in render_paths.items():
            print(f"  {name}: {output_path}")
    else:
        print("no render figures were produced")
except Exception:
    traceback.print_exc()
